# 현재 지금 개발PC(ram 16g, cpu 환경에서 ) 에서 llama3 사용했습니다.
#qwen, gemma등 llm model 이 있지만 그중 llama3가 그중 제일 성능이 났다고해서 사용했습니다. 
# 처음 설치후에 간단한 질문을 날렸는데 대답이나 속도나 비슷했다. 일단 gpt가 추천해주는  llm을 사용해보고 나중에변경해보겠다.
#문서의 구조화, chunnk, metadata, 프롬프트, 데이터 튜닝단계에서 테스트를 해보겠다.

# llama3 말고도 많지만
# mistral → 속도
# qwen → 한국어
# gemma → 가벼움

# ollama pull llama3:8b
# ollama pull qwen2.5:7b
# ollama pull mistral
# ollama pull gemma2:2b

# 2. llm = ChatOllama(model="llama3:8b-instruct-q4")

# 임베딩은 OllamaEmbeddings사용했습니다.
# embedding = OllamaEmbeddings(model="nomic-embed-text")



#현재 개발용 PC는 CPU베이스라 당연히 느려질수 밖에 없어서  Chunk크기, 검색개수(k), 문서개수가 더 중요합니다.
#그래서 아래 llm, embedding, retriever로 세팅해서 사용했습니다.

#llm = ChatOllama(model="llama3:8b-instruct-q4")

#embedding = OllamaEmbeddings(model="nomic-embed-text")

#retriever = db.as_retriever(search_kwargs={"k": 3})

#chuck, mestadata, 문서쪼개기, 문서 구조화yaml, json 변환에 대한 모델링에 집중했습니다.
#양자화 = 모델을 "가볍게 압축"하는 기술, 모델을 작게 만드는 기술, 성능은 조금 포기하고, 속도와 메모리를 높이는걸로 알고있습니다.

#ollama pull llama3

#(2) 모델 테스트하고 싶을 때
#ollama run llama3

#(1) Ollama 안 떠있을 때
#ollama serve

#ollama show llama3:8b


#1. 소득공제 템플릿작성 2. AI로 Yaml작성, 현업도 가능 3. DB입력을 위해 json파일 변환하여 저장 4. metstadata 저장
#2. dictionary, prompt 작성, 질문 llm 전달 => db에서 retriever => llm에 전달
#3. batch flow chart
#4. 테이블 관계도 
#5. 청구 그래프
#6. 출처도 추가

#소득공제, 청구 업무 확인가능

#배치플로우를 보면서 전체적인 배치 흐름도를 파악할 수 있습니다. 해당 배치 id 를 가지고 컨트롤엠에서 진행중인지 확인 가능합니다.
# - 생성형 AI에서 실시간 배치실행을 보려면 컨트롤엠에서  API제공하거나 컨트롤엠 로그를 읽고 처리가능하지만 현재 로컬개발에서는 가능하지 않습니다.

#테이블리니지는 현재 어떻게 테이블 관계도를 보수있느지 확인가능하고 향후 유지보수중에서 테이블 삭제시 영향도 분석도 가능합니다.

# 청구데이터 차트 확인



응. 지금 네 최종 JSON 구조 때문에 성능이 크게 떨어지지는 않아.
대회 관점에서는 “Chroma에 들어가느냐”보다 “어떻게 쪼개고, 무엇을 metadata로 넣고, 검색 후 어떻게 답하게 하느냐”가 훨씬 더 중요해.

정리하면, 네 구조의 장단점은 이래.

좋은 점:

page_content와 metadata가 같이 있어서 Chroma 적재에 바로 쓸 수 있음.
doc_id, domain, section, customer, batch_id, tables, tags 같은 필드가 있어서 필터 검색에 유리함.
배치/장애/개요가 분리돼 있어서 문서 단위가 너무 크지 않음. 이건 검색 정확도에 꽤 도움 됨.

성능 차이가 날 수 있는 부분:

지금은 문서 1개가 통으로 page_content로 들어가니까, 본문이 더 길어지면 chunk 분할이 필요해질 수 있음.
tables, related_docs, tags를 metadata에만 넣고 본문에는 거의 안 풀어 쓰면, 순수 유사도 검색만 할 때는 본문에 없는 키워드 검색이 약할 수 있음.
배치상세, 공통배치, 장애처리처럼 비슷한 문서가 많아지면, 질문 재작성이나 metadata 필터 없이 검색할 때 상위 결과가 섞일 수 있음.

대회에서 실제로 더 중요한 순서는 보통 이거야:

chunk 전략
질문 리라이팅
metadata 필터 활용
임베딩 모델 선택
JSON/YAML 형식

즉,
JSON 포맷 자체 때문에 성능 차이가 나는 건 거의 없고,
성능 차이는 주로 아래에서 난다.

예를 들어:

"고객사1 파일 전송 배치"라고 물었는데
그냥 전체 검색만 하면 비슷한 고객사2 문서도 섞일 수 있음
filter={"customer":"고객사1"} 넣으면 훨씬 좋아짐
"COM_Client001 다음 배치" 같은 질문은
batch_id, prev_batch, next_batch를 metadata에 넣어둔 지금 구조가 유리함.

내 판단으로는:

지금 JSON 구조는 대회용으로 충분히 괜찮다
다만 점수 차이는 JSON 구조가 아니라 retrieval 설계에서 날 가능성이 큼

추천은 이거야.

지금 구조 유지해도 됨.
대신 아래 3가지만 하면 체감 성능이 더 좋아질 거야.

page_content 앞에 메타 일부를 붙여서 저장
[업무명] 소득공제
[고객사] 고객사1
[섹션] 배치상세
[배치ID] TAX-C1-BATCH-004

고객사1 최종 결과 데이터를 기준으로 전달 파일을 생성한다...
검색할 때 metadata 필터 같이 사용
results = db.similarity_search(
    "고객사1 파일 생성 배치",
    k=3,
    filter={"customer": "고객사1"}
)
긴 문서는 chunk 분할
지금은 문서 수가 적어서 괜찮지만
나중에 문서가 길어지면 chunk_size 400~800 정도로 쪼개는 게 좋음

한 줄로 말하면:
네 최종 JSON 파일은 구조상 문제 없고, 대회 성능도 충분히 나올 수 있다. 점수 차이는 JSON보다 검색 설계에서 난다.

원하면 내가 네 현재 JSON 기준으로 대회용 retrieval 최적화 버전 ingest.py로 다시 정리해줄게.

Remove-Item -Recurse -Force .\chroma

In [ ]:
#기본적재 :python ingest.py
#테스트만 : python ingest.py --test
#적재후 테스트 : python ingest.py --ingest-and-test
#출처 엄격하게 python ingest.py --strict
#출처 바꿔서 : python ingest.py --file ./my_document_list.json
#python ingest.py --file ./my_document_list.json --ingest-and-test --strict